<a href="https://colab.research.google.com/github/Sebastian771-177/SENTINEL/blob/main/Sentinel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install anthropic requests --quiet

In [11]:
import anthropic
import requests
import json
import re
import time
from datetime import datetime, UTC
from IPython.display import display, HTML
from google.colab import userdata

In [12]:
try:
    ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY")
    VT_KEY = userdata.get("VIRUSTOTAL_API_KEY")
    ABUSEIPDB_KEY = userdata.get("ABUSEIPDB_API_KEY")
except Exception:
    import os
    ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    VT_KEY = os.environ.get("VIRUSTOTAL_API_KEY", "")
    ABUSEIPDB_KEY = os.environ.get("ABUSEIPDB_API_KEY", "")

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

In [13]:
def detect_ioc_type(ioc: str) -> str:
    """Auto-detect whether input is an IP, domain, hash, or email."""
    ioc = ioc.strip()

    # IPv4
    ip_pattern = r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$"
    if re.match(ip_pattern, ioc):
        return "ip"

    # MD5, SHA1, SHA256
    if re.match(r"^[a-fA-F0-9]{32}$", ioc):
        return "hash_md5"
    if re.match(r"^[a-fA-F0-9]{40}$", ioc):
        return "hash_sha1"
    if re.match(r"^[a-fA-F0-9]{64}$", ioc):
        return "hash_sha256"

    # Email
    if re.match(r"^[^@]+@[^@]+\.[^@]+$", ioc):
        return "email"

    # Domain (fallback)
    if re.match(r"^([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}$", ioc):
        return "domain"

    return "unknown"

In [14]:
def query_virustotal(ioc: str, ioc_type: str) -> dict:
    """Query VirusTotal for IP, domain, or hash reputation."""
    if not VT_KEY:
        return {"error": "No VirusTotal API key provided", "mock": True}

    headers = {"x-apikey": VT_KEY}

    if ioc_type == "ip":
        url = f"https://www.virustotal.com/api/v3/ip_addresses/{ioc}"
    elif ioc_type == "domain":
        url = f"https://www.virustotal.com/api/v3/domains/{ioc}"
    elif ioc_type in ("hash_md5", "hash_sha1", "hash_sha256"):
        url = f"https://www.virustotal.com/api/v3/files/{ioc}"
    else:
        return {"error": "Unsupported IOC type for VirusTotal"}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            stats = data.get("data", {}).get("attributes", {}).get("last_analysis_stats", {})
            return {
                "malicious": stats.get("malicious", 0),
                "suspicious": stats.get("suspicious", 0),
                "harmless": stats.get("harmless", 0),
                "undetected": stats.get("undetected", 0),
                "total": sum(stats.values()),
                "reputation": data.get("data", {}).get("attributes", {}).get("reputation", "N/A"),
            }
        elif response.status_code == 404:
            return {"error": "IOC not found in VirusTotal database"}
        else:
            return {"error": f"VirusTotal returned status {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}


def query_abuseipdb(ip: str) -> dict:
    """Query AbuseIPDB for IP reputation."""
    if not ABUSEIPDB_KEY:
        return {"error": "No AbuseIPDB API key provided", "mock": True}

    headers = {"Key": ABUSEIPDB_KEY, "Accept": "application/json"}
    params = {"ipAddress": ip, "maxAgeInDays": 90, "verbose": True}

    try:
        response = requests.get(
            "https://api.abuseipdb.com/api/v2/check",
            headers=headers,
            params=params,
            timeout=10,
        )
        if response.status_code == 200:
            data = response.json().get("data", {})
            return {
                "abuse_confidence_score": data.get("abuseConfidenceScore", 0),
                "total_reports": data.get("totalReports", 0),
                "country": data.get("countryCode", "Unknown"),
                "isp": data.get("isp", "Unknown"),
                "domain": data.get("domain", "Unknown"),
                "is_tor": data.get("isTor", False),
                "is_whitelisted": data.get("isWhitelisted", False),
                "usage_type": data.get("usageType", "Unknown"),
            }
        else:
            return {"error": f"AbuseIPDB returned status {response.status_code}"}
    except Exception as e:
        return {"error": str(e)}

In [15]:
MOCK_VT_DATA = {
    "ip": {"malicious": 12, "suspicious": 3, "harmless": 45, "undetected": 20, "total": 80, "reputation": -15},
    "domain": {"malicious": 8, "suspicious": 2, "harmless": 30, "undetected": 15, "total": 55, "reputation": -8},
    "hash_md5": {"malicious": 52, "suspicious": 5, "harmless": 0, "undetected": 8, "total": 65, "reputation": -50},
    "hash_sha1": {"malicious": 52, "suspicious": 5, "harmless": 0, "undetected": 8, "total": 65, "reputation": -50},
    "hash_sha256": {"malicious": 52, "suspicious": 5, "harmless": 0, "undetected": 8, "total": 65, "reputation": -50},
    "email": {"malicious": 3, "suspicious": 1, "harmless": 10, "undetected": 5, "total": 19, "reputation": -3},
}

MOCK_ABUSEIPDB_DATA = {
    "abuse_confidence_score": 87,
    "total_reports": 143,
    "country": "RU",
    "isp": "Hosting Provider LLC",
    "domain": "suspicious-host.ru",
    "is_tor": False,
    "is_whitelisted": False,
    "usage_type": "Data Center/Web Hosting/Transit",
}

MOCK_EMAIL_ANALYSIS = {
    "sender_domain": "paypa1-secure.com",
    "spoofed_brand": "PayPal",
    "urgency_indicators": ["Your account will be suspended", "Act immediately"],
    "suspicious_links": ["http://paypa1-secure.com/login"],
    "attachments": [],
}

In [16]:
def analyze_email_with_claude(email_text: str) -> dict:
    """Use Claude to analyze a suspicious email for phishing indicators."""
    prompt = f"""You are a senior SOC analyst specializing in phishing detection.
Analyze the following suspicious email and extract structured threat intelligence.

EMAIL:
\"\"\"
{email_text}
\"\"\"

Return ONLY a valid JSON object with NO additional text or markdown:
{{
  "verdict": "MALICIOUS|SUSPICIOUS|CLEAN",
  "confidence": "HIGH|MEDIUM|LOW",
  "phishing_indicators": ["list of specific phishing red flags found"],
  "spoofed_brand": "brand being impersonated or null",
  "sender_analysis": "analysis of the sender domain and address",
  "link_analysis": "analysis of any URLs found",
  "social_engineering_tactics": ["list of manipulation techniques used"],
  "recommended_actions": ["2-3 specific analyst next steps"],
  "summary": "2-3 sentence plain-English verdict for a SOC report"
}}"""

    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = message.content[0].text.strip().replace("```json", "").replace("```", "")
    return json.loads(raw)


def synthesize_ioc_with_claude(ioc: str, ioc_type: str, vt_data: dict, abuse_data: dict = None) -> dict:
    """Use Claude to synthesize threat intel API results into an analyst report."""

    abuse_section = ""
    if abuse_data:
        abuse_section = f"\nABUSEIPDB DATA:\n{json.dumps(abuse_data, indent=2)}"

    prompt = f"""You are a senior SOC analyst. Synthesize the following threat intelligence
data into a structured analyst report.

IOC: {ioc}
TYPE: {ioc_type}

VIRUSTOTAL DATA:
{json.dumps(vt_data, indent=2)}
{abuse_section}

Return ONLY a valid JSON object with NO additional text or markdown:
{{
  "verdict": "MALICIOUS|SUSPICIOUS|CLEAN|UNKNOWN",
  "severity": "CRITICAL|HIGH|MEDIUM|LOW",
  "confidence": "HIGH|MEDIUM|LOW",
  "threat_type": "concise threat category (e.g. C2 Server, Malware Distribution, Phishing Infrastructure)",
  "summary": "2-3 sentence plain-English analyst verdict",
  "key_findings": ["3-4 most important findings from the data"],
  "recommended_actions": ["2-3 specific analyst next steps"],
  "false_positive_likelihood": "HIGH|MEDIUM|LOW",
  "mitre_ttps": ["relevant MITRE ATT&CK technique names"]
}}"""

    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = message.content[0].text.strip().replace("```json", "").replace("```", "")
    return json.loads(raw)

In [17]:
VERDICT_COLORS = {
    "MALICIOUS": ("#ff2d55", "#2a0a10"),
    "SUSPICIOUS": ("#ff9f0a", "#2a1a00"),
    "CLEAN":      ("#30d158", "#0a1f10"),
    "UNKNOWN":    ("#636366", "#1a1a1a"),
}

SEVERITY_COLORS = {
    "CRITICAL": "#ff2d55",
    "HIGH":     "#ff9f0a",
    "MEDIUM":   "#ffd60a",
    "LOW":      "#30d158",
}


def pill(text, color="#aaa", bg="rgba(255,255,255,0.08)"):
    return (
        f'<span style="display:inline-block;padding:2px 10px;border-radius:20px;'
        f'font-size:11px;font-weight:600;background:{bg};color:{color};'
        f'border:1px solid {color}40;margin:2px;">{text}</span>'
    )


def section(title, items):
    if not items:
        return ""
    if isinstance(items, list):
        content = "".join(
            f"<li style='margin:3px 0;color:#ccc;font-size:13px;'>{i}</li>"
            for i in items
        )
        content = f"<ul style='margin:6px 0 0 16px;padding:0;'>{content}</ul>"
    else:
        content = f"<p style='margin:6px 0 0;color:#ccc;font-size:13px;line-height:1.5;'>{items}</p>"
    return (
        f'<div style="margin-top:14px;">'
        f'<div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;'
        f'text-transform:uppercase;margin-bottom:2px;">{title}</div>'
        f"{content}</div>"
    )


def render_report(ioc: str, ioc_type: str, analysis: dict, vt_data: dict = None, abuse_data: dict = None) -> str:
    verdict = analysis.get("verdict", "UNKNOWN")
    verdict_color, verdict_bg = VERDICT_COLORS.get(verdict, ("#636366", "#1a1a1a"))
    severity = analysis.get("severity", "MEDIUM")
    severity_color = SEVERITY_COLORS.get(severity, "#ffd60a")

    vt_section = ""
    if vt_data and "error" not in vt_data:
        vt_section = f"""
        <div style="background:#111;border-radius:6px;padding:12px;margin-top:14px;">
          <div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;
                      text-transform:uppercase;margin-bottom:8px;">🛡 VirusTotal Results</div>
          <div style="display:flex;gap:16px;flex-wrap:wrap;">
            <div style="text-align:center;">
              <div style="font-size:20px;font-weight:700;color:#ff2d55;">{vt_data.get('malicious',0)}</div>
              <div style="font-size:10px;color:#555;">Malicious</div>
            </div>
            <div style="text-align:center;">
              <div style="font-size:20px;font-weight:700;color:#ff9f0a;">{vt_data.get('suspicious',0)}</div>
              <div style="font-size:10px;color:#555;">Suspicious</div>
            </div>
            <div style="text-align:center;">
              <div style="font-size:20px;font-weight:700;color:#30d158;">{vt_data.get('harmless',0)}</div>
              <div style="font-size:10px;color:#555;">Harmless</div>
            </div>
            <div style="text-align:center;">
              <div style="font-size:20px;font-weight:700;color:#eee;">{vt_data.get('total',0)}</div>
              <div style="font-size:10px;color:#555;">Total Engines</div>
            </div>
          </div>
        </div>"""

    abuse_section_html = ""
    if abuse_data and "error" not in abuse_data:
        score = abuse_data.get("abuse_confidence_score", 0)
        score_color = "#ff2d55" if score > 50 else "#ff9f0a" if score > 20 else "#30d158"
        abuse_section_html = f"""
        <div style="background:#111;border-radius:6px;padding:12px;margin-top:14px;">
          <div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;
                      text-transform:uppercase;margin-bottom:8px;">🚨 AbuseIPDB Results</div>
          <div style="display:flex;gap:16px;flex-wrap:wrap;align-items:center;">
            <div style="text-align:center;">
              <div style="font-size:20px;font-weight:700;color:{score_color};">{score}%</div>
              <div style="font-size:10px;color:#555;">Abuse Score</div>
            </div>
            <div style="font-size:12px;color:#888;line-height:1.8;">
              Reports: <span style="color:#ccc;">{abuse_data.get('total_reports',0)}</span><br>
              Country: <span style="color:#ccc;">{abuse_data.get('country','?')}</span><br>
              ISP: <span style="color:#ccc;">{abuse_data.get('isp','?')}</span><br>
              Tor Exit: <span style="color:#ccc;">{'Yes' if abuse_data.get('is_tor') else 'No'}</span>
            </div>
          </div>
        </div>"""

    ttp_pills = "".join(pill(t, "#bf5fff", "rgba(191,95,255,0.1)") for t in analysis.get("mitre_ttps", []))

    html = f"""
    <div style="font-family:'Courier New',monospace;background:#0d0d0d;
                border:1px solid {verdict_color}40;border-left:4px solid {verdict_color};
                border-radius:8px;padding:20px;margin:12px 0;
                box-shadow:0 4px 24px rgba(0,0,0,0.5);">

      <!-- Header -->
      <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:8px;">
        <div>
          <div style="font-size:11px;color:#555;letter-spacing:0.15em;margin-bottom:4px;">
            SENTINEL · {datetime.now(UTC).strftime('%Y-%m-%d %H:%M UTC')}
          </div>
          <div style="font-size:18px;font-weight:700;color:#eee;word-break:break-all;">{ioc}</div>
          <div style="font-size:12px;color:#888;margin-top:2px;">
            Type: <span style="color:#aaa;">{ioc_type.upper()}</span>
          </div>
        </div>
        <div style="display:flex;flex-direction:column;align-items:flex-end;gap:6px;">
          <span style="padding:6px 16px;border-radius:4px;font-size:13px;font-weight:800;
                       letter-spacing:0.1em;background:{verdict_bg};color:{verdict_color};
                       border:1px solid {verdict_color};">● {verdict}</span>
          <span style="font-size:12px;color:{severity_color};font-weight:700;">⚠ {severity}</span>
          <span style="font-size:11px;color:#555;">Confidence:
            <span style="color:#aaa;">{analysis.get('confidence','?')}</span>
          </span>
        </div>
      </div>

      <div style="border-top:1px solid #222;margin:14px 0;"></div>

      {section("📋 Analyst Summary", analysis.get("summary", ""))}
      {vt_section}
      {abuse_section_html}
      {section("🔍 Key Findings", analysis.get("key_findings", analysis.get("phishing_indicators", [])))}
      {('<div style="margin-top:14px;"><div style="font-size:10px;letter-spacing:0.12em;color:#666;font-weight:700;text-transform:uppercase;margin-bottom:6px;">⚔ MITRE ATT&CK TTPs</div>' + ttp_pills + '</div>') if ttp_pills else ''}
      {section("✅ Recommended Actions", analysis.get("recommended_actions", []))}
      {section("⚠ Social Engineering Tactics", analysis.get("social_engineering_tactics", []))}
      <div style="margin-top:14px;font-size:11px;color:#444;">
        False Positive Likelihood: <span style="color:#666;">{analysis.get('false_positive_likelihood','MEDIUM')}</span>
      </div>
    </div>
    """
    return html


def render_header():
    return """
    <div style="font-family:'Courier New',monospace;background:#050505;
                border:1px solid #1a1a1a;border-radius:8px;padding:24px;margin-bottom:8px;">
      <div style="color:#5fb3ff;font-size:22px;font-weight:700;letter-spacing:0.15em;
                  text-shadow:0 0 20px rgba(95,179,255,0.5);">
        ◈ SENTINEL
      </div>
      <div style="color:#444;font-size:11px;letter-spacing:0.3em;margin-top:4px;">
        AI-POWERED SOC ASSISTANT · IOC TRIAGE & ANALYSIS
      </div>
    </div>
    """

In [18]:
def analyze_ioc(ioc: str, use_mock: bool = False) -> dict:
    """
    Analyze a single IOC (IP, domain, hash, or email).

    Args:
        ioc: The indicator of compromise to analyze
        use_mock: If True, uses mock data instead of real API calls
    Returns:
        Full analysis report as a dict
    """
    ioc = ioc.strip()
    ioc_type = detect_ioc_type(ioc)

    print(f"  → Detected type: {ioc_type.upper()}")

    vt_data = None
    abuse_data = None

    if ioc_type == "email":
        print("  → Running Claude email analysis...")
        if use_mock:
            analysis = {
                "verdict": "MALICIOUS",
                "confidence": "HIGH",
                "severity": "HIGH",
                "phishing_indicators": [
                    "Sender domain does not match claimed brand",
                    "Urgency language designed to bypass critical thinking",
                    "Suspicious link redirecting to non-brand domain",
                    "Generic greeting instead of personalized salutation",
                ],
                "spoofed_brand": "PayPal",
                "sender_analysis": "Domain paypa1-secure.com registered 3 days ago, unrelated to PayPal",
                "link_analysis": "URL uses typosquatted domain with HTTP (not HTTPS)",
                "social_engineering_tactics": ["Fear/urgency", "Authority impersonation", "Account suspension threat"],
                "recommended_actions": [
                    "Block sender domain at email gateway",
                    "Submit URL to Google Safe Browsing",
                    "Alert users who may have received similar emails",
                ],
                "summary": "This email is a PayPal phishing attempt using a typosquatted domain and urgency tactics to steal credentials. The sender domain was registered recently and has no affiliation with PayPal. Recommend immediate blocking.",
                "false_positive_likelihood": "LOW",
                "mitre_ttps": ["Phishing", "Spearphishing Link", "User Execution"],
            }
        else:
            analysis = analyze_email_with_claude(ioc)
        analysis["severity"] = analysis.get("severity", "HIGH")
        analysis["false_positive_likelihood"] = analysis.get("false_positive_likelihood", "LOW")
        analysis["mitre_ttps"] = analysis.get("mitre_ttps", ["Phishing"])

    else:
        # Query VirusTotal
        print("  → Querying VirusTotal...")
        vt_data = MOCK_VT_DATA.get(ioc_type, {}) if use_mock else query_virustotal(ioc, ioc_type)

        # Query AbuseIPDB for IPs
        if ioc_type == "ip":
            print("  → Querying AbuseIPDB...")
            abuse_data = MOCK_ABUSEIPDB_DATA if use_mock else query_abuseipdb(ioc)

        # Synthesize with Claude
        print("  → Synthesizing with Claude...")
        if use_mock:
            analysis = {
                "verdict": "MALICIOUS",
                "severity": "HIGH",
                "confidence": "HIGH",
                "threat_type": "C2 Server / Malware Infrastructure",
                "summary": f"This {ioc_type} shows strong indicators of malicious activity with significant detections across multiple threat intelligence engines. The high abuse confidence score and association with suspicious hosting infrastructure suggest active use in cybercriminal operations.",
                "key_findings": [
                    f"{vt_data.get('malicious', 0)} of {vt_data.get('total', 0)} VirusTotal engines flagged as malicious",
                    "Associated with data center hosting commonly used for C2 infrastructure",
                    "No legitimate business justification found for this indicator",
                    "Recently reported by multiple threat intelligence sources",
                ],
                "recommended_actions": [
                    "Block immediately at firewall and proxy",
                    "Search SIEM for any internal hosts communicating with this indicator",
                    "Submit to your threat intel platform for tracking",
                ],
                "false_positive_likelihood": "LOW",
                "mitre_ttps": ["Command and Control", "Exfiltration Over C2 Channel", "Ingress Tool Transfer"],
            }
        else:
            analysis = synthesize_ioc_with_claude(ioc, ioc_type, vt_data, abuse_data)

    return {
        "ioc": ioc,
        "ioc_type": ioc_type,
        "analysis": analysis,
        "vt_data": vt_data,
        "abuse_data": abuse_data,
    }


def run_sentinel(iocs: list, use_mock: bool = False):
    """
    Main pipeline — analyze a list of IOCs and render the dashboard.

    Args:
        iocs: List of IOC strings to analyze
        use_mock: Set True to run without API keys (demo mode)
    """
    display(HTML(render_header()))
    print(f"[SENTINEL] Analyzing {len(iocs)} indicator(s)...\n")

    results = []
    for i, ioc in enumerate(iocs):
        print(f"[{i+1}/{len(iocs)}] Analyzing: {ioc}")
        try:
            result = analyze_ioc(ioc, use_mock=use_mock)
            results.append(result)
            verdict = result["analysis"].get("verdict", "?")
            print(f"  ✓ Verdict: {verdict}\n")
            display(HTML(render_report(
                result["ioc"],
                result["ioc_type"],
                result["analysis"],
                result["vt_data"],
                result["abuse_data"],
            )))
        except Exception as e:
            print(f"  ✗ Error: {e}\n")

        if i < len(iocs) - 1:
            time.sleep(1)

    print(f"[SENTINEL] Analysis complete. {len(results)} report(s) generated.")
    return results


In [19]:
SAMPLE_IOCS = [
    # Suspicious IP
    "185.220.101.45",

    # Suspicious domain
    "paypa1-secure-login.com",

    # Known malware hash (MD5)
    "44d88612fea8a8f36de82e1278abb02f",

    # Phishing email (paste full email text for real analysis)
    """From: security@paypa1-secure.com
Subject: URGENT: Your account has been limited
Dear Customer,
Your PayPal account has been limited due to suspicious activity.
Click here immediately to restore access: http://paypa1-secure.com/login
Failure to act within 24 hours will result in permanent suspension.
PayPal Security Team""",
]

if __name__ == "__main__":
    # Set use_mock=True to run without API keys
    # Set use_mock=False once you have real API keys configured
    reports = run_sentinel(SAMPLE_IOCS, use_mock=True)

    # Save results
    with open("sentinel_output.json", "w") as f:
        json.dump(reports, f, indent=2, default=str)
    print("\n[SENTINEL] Reports saved to sentinel_output.json")


[SENTINEL] Analyzing 4 indicator(s)...

[1/4] Analyzing: 185.220.101.45
  → Detected type: IP
  → Querying VirusTotal...
  → Querying AbuseIPDB...
  → Synthesizing with Claude...
  ✓ Verdict: MALICIOUS



[2/4] Analyzing: paypa1-secure-login.com
  → Detected type: DOMAIN
  → Querying VirusTotal...
  → Synthesizing with Claude...
  ✓ Verdict: MALICIOUS



[3/4] Analyzing: 44d88612fea8a8f36de82e1278abb02f
  → Detected type: HASH_MD5
  → Querying VirusTotal...
  → Synthesizing with Claude...
  ✓ Verdict: MALICIOUS



[4/4] Analyzing: From: security@paypa1-secure.com
Subject: URGENT: Your account has been limited
Dear Customer,
Your PayPal account has been limited due to suspicious activity.
Click here immediately to restore access: http://paypa1-secure.com/login
Failure to act within 24 hours will result in permanent suspension.
PayPal Security Team
  → Detected type: EMAIL
  → Running Claude email analysis...
  ✓ Verdict: MALICIOUS



[SENTINEL] Analysis complete. 4 report(s) generated.

[SENTINEL] Reports saved to sentinel_output.json
